# Volume 4, Chapter 8: The Fourier transform and Poisson summation (Python companion)

Exercises first, solutions at the end. We check transforms numerically to 30 digits, watch Poisson summation turn a slow series into a fast one, verify Gauss's sign on two thousand sums, see aliasing happen, solve the heat equation by FFT, and measure how close functions come to the Heisenberg bound. Requirements: numpy and mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P8.1 (★) Transforms to thirty digits.** With the convention $\hat f(\xi)=\int f(x)e^{-2\pi ix\xi}\,dx$, check numerically the transforms of $e^{-\pi x^2}$ (Theorem 8.3), $e^{-2\pi|x|}$ and $\frac1{1+x^2}$ (Exercise 8.1), and $\operatorname{sech}(\pi x)$. Oscillatory integrals over $[0,\infty)$ need care: why does a plain quadrature fail, and what does `quadosc` do instead?

**P8.2 (★) Poisson summation accelerates.** Compute $\theta(t)=\sum_n e^{-\pi n^2t}$ at $t=0.001$ directly, with more and more terms, and then through Corollary 8.10. Also check $\sum_n\frac1{n^2+a^2}=\frac\pi a\coth(\pi a)$ (Exercise 8.3) and $\sum_n\frac1{(x+n)^2}=\frac{\pi^2}{\sin^2\pi x}$ (Proposition 8.11).

**P8.3 (★) Gauss's sign.** Compute the quadratic Gauss sums $G(n)=\sum_{k=0}^{n-1}e^{2\pi ik^2/n}$ for all $n\le2000$ and compare with the formula of Theorem 8.14. Look at the four cases $n\bmod4$.

**P8.4 (★) CS application: aliasing.** Sample $\sin(2\pi ft)$ ten times per unit of time, for several frequencies $f$, and find the frequency the samples appear to have (the peak of their DFT). Compare with Theorem 8.15 and Figure 8.3. Which frequencies are reproduced faithfully?

**P8.5 (★★) CS application: the heat equation, spectral against finite differences.** On the circle of length $1$, solve $u_t=u_{xx}$ from a box-shaped initial temperature, once exactly by FFT (each Fourier mode decays like $e^{-4\pi^2k^2t}$, Proposition 8.8 and Exercise 6.3) and once with the explicit scheme $u\leftarrow u+\frac{\Delta t}{\Delta x^2}(u_{j+1}-2u_j+u_{j-1})$. Try $\Delta t=0.25\,\Delta x^2$, $0.5\,\Delta x^2$ and $0.51\,\Delta x^2$. Explain what you see with Fourier analysis.

**P8.6 (★★) The Heisenberg ratio.** For $f$ in $\{e^{-\pi x^2},\ \operatorname{sech}(\pi x),\ xe^{-\pi x^2}\}$, compute the ratio of $\int x^2|f|^2\int\xi^2|\hat f|^2$ to the lower bound $\frac{\|f\|_2^4}{16\pi^2}$ of Theorem 8.16. Which function attains the bound? (The transform of $xe^{-\pi x^2}$ is $-i\xi e^{-\pi\xi^2}$, Exercise 8.2; the factor $-i$ does not change the modulus.)

## Solutions

### P8.1

In [ ]:
from mpmath import mp, mpf, quad, quadosc, exp, cos, sin, pi, inf, sqrt, sech, coth, nsum, cot
mp.dps = 30
def ft(f, xi):                     # transform of an even function: 2 int_0^inf f(x) cos(2 pi x xi) dx
    if xi == 0:
        return 2 * quad(f, [0, inf])
    return 2 * quadosc(lambda x: f(x) * cos(2 * pi * x * xi), [0, inf], omega=2 * pi * xi)
tests = [
    ("Gaussian e^{-pi x^2}  ", lambda x: exp(-pi * x**2),   lambda s: exp(-pi * s**2)),
    ("e^{-2 pi |x|}         ", lambda x: exp(-2 * pi * x),  lambda s: 1 / (pi * (1 + s**2))),
    ("1/(1+x^2)             ", lambda x: 1 / (1 + x**2),    lambda s: pi * exp(-2 * pi * s)),
    ("sech(pi x)            ", lambda x: sech(pi * x),      lambda s: sech(pi * s)),
]
for name, f, F in tests:
    errs = [abs(ft(f, mpf(s)) - F(mpf(s))) for s in ("0", "0.3", "1.7")]
    print(name, "max error at xi = 0, 0.3, 1.7:", mp.nstr(max(errs), 3))

All four agree to about $30$ digits. For an even $f$, $\hat f(\xi)=2\int_0^\infty f(x)\cos(2\pi x\xi)\,dx$. For $\frac1{1+x^2}$ the integrand decays slowly and changes sign infinitely often, so a plain quadrature on $[0,\infty)$ sees huge cancellations it cannot resolve. `quadosc` integrates between consecutive zeros of the cosine and adds up the resulting alternating series with a convergence accelerator. Notice that $\operatorname{sech}(\pi x)$ is its own transform, like the Gaussian: the transform has more fixed points than one might guess.

### P8.2

In [ ]:
def theta(t, terms):
    return 1 + 2 * sum(exp(-pi * n * n * t) for n in range(1, terms + 1))
t = mpf("0.001")
for terms in (10, 50, 100, 200):
    print(f"theta(0.001) with {terms:3d} terms: {mp.nstr(theta(t, terms), 25)}")
print(f"sqrt(1/t) * theta(1/t), 2 terms : {mp.nstr(sqrt(1 / t) * theta(1 / t, 2), 25)}")
a = mpf("0.7")
print("sum 1/(n^2+a^2):", mp.nstr(nsum(lambda n: 1 / (n**2 + a**2), [-inf, inf]), 20), "  (pi/a) coth(pi a):", mp.nstr(pi / a * coth(pi * a), 20))
x = mpf("0.3")
print("sum 1/(x+n)^2  :", mp.nstr(nsum(lambda n: 1 / (x + n)**2, [-inf, inf]), 20), "  pi^2/sin^2(pi x):", mp.nstr(pi**2 / sin(pi * x)**2, 20))

The direct series needs about $100$ terms, because $e^{-\pi n^2/1000}$ is small only once $n$ is well beyond $\sqrt{1000/\pi}\approx18$. Through $\theta(t)=t^{-1/2}\theta(1/t)$ two terms already give $25$ digits, since $e^{-1000\pi}$ is about $10^{-1364}$. This is the general lesson of Poisson summation: a function that is wide in $x$ is narrow in $\xi$, so a slowly converging sum becomes a rapidly converging one. Chapter 10 uses exactly this trick to continue the zeta function.

### P8.3

In [ ]:
import cmath, math
def G(n):
    return sum(cmath.exp(2j * math.pi * k * k / n) for k in range(n))
def formula(n):
    return (1 + 1j) * (1 + (1j) ** (-n)) / 2 * math.sqrt(n)
worst = max(abs(G(n) - formula(n)) for n in range(1, 2001))
print("max |G(n) - formula| for n <= 2000:", f"{worst:.1e}")
for n in (5, 6, 7, 8, 101, 103):
    g = G(n)
    print(f"n = {n:3d} (n mod 4 = {n % 4}): G(n) = {g.real:+9.5f} {g.imag:+9.5f}i,  sqrt(n) = {math.sqrt(n):.5f}")

The formula $G(n)=\frac{(1+i)(1+i^{-n})}2\sqrt n$ holds for every $n\le2000$ up to rounding error. It gives $\sqrt n$ for $n\equiv1$, $0$ for $n\equiv2$, $i\sqrt n$ for $n\equiv3$, and $(1+i)\sqrt n$ for $n\equiv0\pmod4$. The modulus is easy (Volume 0); the sign is what Gauss struggled with for four years, and Theorem 8.14 obtains it in one stroke from Poisson summation.

### P8.4

In [ ]:
import numpy as np
R, N = 10, 1000                      # 10 samples per unit, 100 units of signal
t = np.arange(N) / R
print(" true f   apparent f (peak of the DFT)   predicted |f - R round(f/R)|")
for f in (1, 3, 4.9, 6, 9, 11, 14.5, 19, 21):
    A = np.abs(np.fft.rfft(np.sin(2 * np.pi * f * t)))
    apparent = np.argmax(A) * R / N
    print(f" {f:5.1f}      {apparent:6.2f}                         {abs(f - R * round(f / R)):6.2f}")

With $R=10$ samples per unit, frequencies below the Nyquist frequency $\frac R2=5$ are reproduced faithfully. Any higher frequency is folded back to $|f-R\cdot\mathrm{round}(f/R)|$: $9$ looks like $1$, $11$ looks like $1$, $14.5$ looks like $4.5$. The samples of $\sin(2\pi\cdot 9t)$ and $\sin(2\pi\cdot 1t)$ differ only by a sign; nothing in the data can tell them apart. This is why audio is low-pass filtered before sampling at $44.1$ kHz, a bit more than twice the $20$ kHz limit of hearing (Theorem 8.15).

### P8.5

In [ ]:
L, M = 1.0, 256                       # heat equation u_t = u_xx on the circle of length 1
x = np.arange(M) / M
u0 = ((x > 0.3) & (x < 0.6)).astype(float)
k = np.fft.fftfreq(M, d=1 / M)        # integer frequencies
def spectral(u0, T):
    return np.fft.ifft(np.fft.fft(u0) * np.exp(-4 * np.pi**2 * k**2 * T)).real
def finite_diff(u0, T, dt):
    u = u0.copy(); dx = 1 / M; steps = int(round(T / dt))
    for _ in range(steps):
        u = u + dt / dx**2 * (np.roll(u, 1) - 2 * u + np.roll(u, -1))
    return u, steps
T = 0.002
ref = spectral(u0, T)
dx = 1 / M
for r in (0.25, 0.5, 0.51):
    dt = r * dx**2
    u, steps = finite_diff(u0, T, dt)
    print(f"finite differences, dt = {r:.2f} dx^2 ({steps:4d} steps): max difference with the spectral solution = {np.abs(u - ref).max():.2e}")
print("total heat conserved by the spectral solution:", np.isclose(ref.sum(), u0.sum()))

At $\Delta t=0.25\,\Delta x^2$ the scheme matches the spectral solution to about $5\cdot10^{-5}$. At $0.5\,\Delta x^2$ it is still bounded, but the error is a hundred times larger. At $0.51\,\Delta x^2$ it explodes. Fourier analysis explains why. The scheme multiplies the mode $e^{2\pi ikx}$ by $1-4r\sin^2(\pi k\Delta x)$ at each step, where $r=\Delta t/\Delta x^2$. For the highest mode the factor is $1-4r$, which has modulus above $1$ exactly when $r>\frac12$. That mode is then amplified at every step, until rounding errors grow into nonsense: this is the stability limit $\Delta t\le\frac{\Delta x^2}2$ (von Neumann analysis). The spectral method has no such limit, because it applies the exact factor $e^{-4\pi^2k^2T}$.

### P8.6

In [ ]:
def heisenberg(f, F):
    a = quad(lambda x: x**2 * abs(f(x))**2, [-inf, inf])
    b = quad(lambda s: s**2 * abs(F(s))**2, [-inf, inf])
    c = quad(lambda x: abs(f(x))**2, [-inf, inf])
    return a * b / c**2 * 16 * pi**2
print("Gaussian    :", mp.nstr(heisenberg(lambda x: exp(-pi * x**2), lambda s: exp(-pi * s**2)), 12))
print("sech(pi x)  :", mp.nstr(heisenberg(lambda x: sech(pi * x), lambda s: sech(pi * s)), 12))
print("x e^{-pi x^2}:", mp.nstr(heisenberg(lambda x: x * exp(-pi * x**2), lambda s: s * exp(-pi * s**2)), 12))

The Gaussian attains the bound exactly, as the equality case of Theorem 8.16 predicts. $\operatorname{sech}(\pi x)$, which is also its own transform, comes within $10\%$ of it. $xe^{-\pi x^2}$ gives exactly $9$: it is the next Hermite function, and the $n$-th Hermite function gives $(2n+1)^2$. In quantum mechanics these are the energy levels of the harmonic oscillator.